# Model Training 03c (Benchmark-Derived)

This notebook is intentionally simple and benchmark-aligned:

1. use only 4 base features (`swir22`, `NDMI`, `MNDWI`, `pet`),
2. train 3 separate Random Forest regressors,
3. evaluate with a random 70/30 split,
4. generate a submission from merged MVP validation parquet.

In [1]:
import os
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
TRAIN_PATH = '../data/interim/water_quality_mvp_baseline.parquet'
VALID_PATH = '../data/interim/water_quality_mvp_validation.parquet'
TEMPLATE_PATH = '../data/raw/submission_template.csv'

FEATURES = ['swir22', 'NDMI', 'MNDWI', 'pet']
TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
META = ['Longitude', 'Latitude', 'Sample Date']

train_df = pd.read_parquet(TRAIN_PATH).copy()
val_df = pd.read_parquet(VALID_PATH).copy()
template_df = pd.read_csv(TEMPLATE_PATH).copy()

required_train = FEATURES + TARGETS
missing_train = [c for c in required_train if c not in train_df.columns]
if missing_train:
    raise RuntimeError(f'Missing training columns: {missing_train}')

missing_val = [c for c in FEATURES if c not in val_df.columns]
if missing_val:
    raise RuntimeError(f'Missing validation columns: {missing_val}')

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Template shape:', template_df.shape)

Train shape: (9319, 10)
Validation shape: (200, 10)
Template shape: (200, 6)


In [3]:
# Median imputation using training statistics
train_medians = train_df[FEATURES].median(numeric_only=True)

X_full = train_df[FEATURES].copy().fillna(train_medians)
Y_full = train_df[TARGETS].copy()
X_sub = val_df[FEATURES].copy().fillna(train_medians)

print('Feature null counts (train):')
print(X_full.isna().sum())
print('\nFeature null counts (validation):')
print(X_sub.isna().sum())

Feature null counts (train):
swir22    0
NDMI      0
MNDWI     0
pet       0
dtype: int64

Feature null counts (validation):
swir22    0
NDMI      0
MNDWI     0
pet       0
dtype: int64


In [4]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def train_model(X_train_scaled, y_train):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    return model


def evaluate_model(model, X_scaled, y_true, dataset_name='Test'):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'\n{dataset_name} Evaluation:')
    print(f'R2: {r2:.3f}')
    print(f'RMSE: {rmse:.3f}')
    return y_pred, r2, rmse


def run_pipeline(X, y, param_name='Parameter'):
    print(f"\n{'='*60}")
    print(f'Training Model for {param_name}')
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    model = train_model(X_train_scaled, y_train)

    _, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, 'Train')
    _, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, 'Test')

    results = {
        'Parameter': param_name,
        'R2_Train': r2_train,
        'RMSE_Train': rmse_train,
        'R2_Test': r2_test,
        'RMSE_Test': rmse_test,
    }
    return model, scaler, pd.DataFrame([results])

In [5]:
models = {}
scalers = {}
results = []

for target in TARGETS:
    model, scaler, res = run_pipeline(X_full, Y_full[target], target)
    models[target] = model
    scalers[target] = scaler
    results.append(res)

results_summary = pd.concat(results, ignore_index=True)
results_summary


Training Model for Total Alkalinity

Train Evaluation:
R2: 0.903
RMSE: 23.124

Test Evaluation:
R2: 0.546
RMSE: 50.877

Training Model for Electrical Conductance

Train Evaluation:
R2: 0.918
RMSE: 98.029

Test Evaluation:
R2: 0.585
RMSE: 220.214

Training Model for Dissolved Reactive Phosphorus

Train Evaluation:
R2: 0.882
RMSE: 17.455

Test Evaluation:
R2: 0.529
RMSE: 35.182


,Parameter,R2_Train,RMSE_Train,R2_Test,RMSE_Test
0,Total Alkalinity,0.903272,23.123742,0.545544,50.876902
1,Electrical Conductance,0.917848,98.028822,0.584648,220.213714
2,Dissolved Reactive Phosphorus,0.882169,17.454757,0.529145,35.181776


In [ ]:
# # Build submission from validation features
# preds = {}
# for target in TARGETS:
#     X_sub_scaled = scalers[target].transform(X_sub)
#     preds[target] = models[target].predict(X_sub_scaled)

# submission_df = pd.DataFrame({
#     'Longitude': template_df['Longitude'].values,
#     'Latitude': template_df['Latitude'].values,
#     'Sample Date': template_df['Sample Date'].values,
#     'Total Alkalinity': preds['Total Alkalinity'],
#     'Electrical Conductance': preds['Electrical Conductance'],
#     'Dissolved Reactive Phosphorus': preds['Dissolved Reactive Phosphorus'],
# })

# submission_df.head()

In [ ]:
# os.makedirs('../data/submission', exist_ok=True)
# stamp = datetime.now().strftime('%Y%m%d_%H%M')
# out_path = f'../data/submission/submission_{stamp}_03c_benchmark_derived.csv'
# submission_df.to_csv(out_path, index=False)
# print('Saved:', out_path)